# Signatures comparison — new sorted-cell test cohort (CHESS-1333 / OD-128)

Companion to the published Figure 4 notebook (`Signatures comparison.ipynb`). The new sorted-cell cohort delivered with [Jira OD-128](https://bostongene.atlassian.net/browse/OD-128) serves as a true held-out **test** cohort, satisfying Reviewer 1's request to "Add new data of sorted cells to cell signature comparison" and the paper Methods commitment to ~75/25 train/test separation.

**Scope:** 16 of 20 FGES. The four rare-GOI FGES — `Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` — are deferred to a separate rare-types notebook that uses 75/25 stratified holdouts on the original cohort (helpers ship in `signature_validation.benchmark.splits`).

**Random-FGES baseline:** v1 random gene lists are reused (loaded from `msigdb_gmt.pkl`) but rescored on the new cohort so ranks stay comparable.

**Where things land:** pickle and SVGs go to `/home/jovyan/SignValArticle/...` with a `_new_cohort` suffix; v1 outputs are not overwritten.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger

from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    MAP_RAW,
    build_mapping,
    intersect_controls_with_cohort,
    load_new_cohort_annotation,
    load_new_cohort_expressions,
)
from signature_validation.benchmark.plotting import (
    plot_sens_spec_scatter,
    plot_signature_heatmap,
    plot_violin_per_source,
)
from signature_validation.benchmark.scoring import (
    compute_mapping_ssgseas,
    compute_out_table,
    fdr_correct_out,
)
from signature_validation.benchmark.signatures import (
    count_random_fges,
    harmonize_gmt_to_index,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.plotting.plotting import cells_p

sns.set_style("white")
plt.rcParams["svg.fonttype"] = "none"

In [3]:
NEW_ANNOT_PATH = Path(
    "/home/jovyan/projects/SignVal/Signature_validation/sorted_cells_to_check_all_annot.tsv"
)

OUTPUT_DIR = Path("/home/jovyan/SignValArticle/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_SSGSEAS_PATH = OUTPUT_DIR / "mapping_ssgseas_new_cohort.pkl"
OUT_TSV_PATH = OUTPUT_DIR / "out_new_cohort.tsv"
HEATMAP_PATH = OUTPUT_DIR / "signature_heatmap_new_cohort.svg"

logger.info("new annotation:    {}", NEW_ANNOT_PATH)
logger.info("mapping_ssgseas:   {}", MAPPING_SSGSEAS_PATH)

2026-07-12 22:43:32.718 | INFO     | __main__:<module>:11 - new annotation:    /home/jovyan/projects/SignVal/Signature_validation/sorted_cells_to_check_all_annot.tsv
2026-07-12 22:43:32.719 | INFO     | __main__:<module>:12 - mapping_ssgseas:   /home/jovyan/SignValArticle/mapping_ssgseas_new_cohort.pkl


In [4]:
public_cells_annot = load_new_cohort_annotation(NEW_ANNOT_PATH)
public_cells_annot["Cell_type"].value_counts()

/home/jovyan/projects/Signature_Validation/src/signature_validation/utils/utils.py:541: DtypeWarning: Columns (4,28,30,31,33,34,35,36,39,41,44,47,52,53,54,55,57,59) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(
2026-07-12 22:43:33.374 | INFO     | signature_validation.benchmark.cohorts:load_new_cohort_annotation:226 - loaded 30418 samples across 217 cell types from /home/jovyan/projects/SignVal/Signature_validation/sorted_cells_to_check_all_annot.tsv


Cell_type
Epithelium              3155
Macrophages             2404
CD4_T_cells             2379
Monocytes               2248
CD8_T_cells             2009
                        ... 
Mature_neutrophils         1
Trophoblast_cells          1
Cytotoxic_NK_cells         1
Intestinal_organoids       1
Schwann_cells              1
Name: count, Length: 217, dtype: int64

In [5]:
from signature_validation.utils.utils import read_expressions

In [6]:
EXPR_S3_PATH = "/uftp2/Databases/Deconvolution/"
public_cells_expr = read_expressions(public_cells_annot, path=EXPR_S3_PATH)
public_cells_expr.shape

no GSE58310 expression
no GSE101993 expression
no GSE141217 expression
no GSE152446 expression
no GSE154122 expression
no GSE177862 expression
no GSE184398 expression
no GSE191279 expression
no GSE193682 expression
no GSE201152 expression
no GSE77312 expression
no GSE83492 expression
no GSE84135 expression
no PRJNA360082 expression
no PRJNA374973 expression
no PRJNA397967 expression
no PRJNA434217 expression
no PRJNA436739 expression
no PRJNA484735 expression
no PRJNA491656 expression
no PRJNA596741 expression
no PRJNA624366 expression
no PRJNA631458 expression
no GSE223806 expression
no GSE171256 expression
no GSE152590 expression
no GSE213696 expression
no GSE173387 expression
no GSE221563 expression
no GSE235755 expression
no GSE162712 expression
no GSE199490 expression
no GSE226249 expression
no GSE247226 expression
no GSE188464 expression
no GSE173635 expression
no GSE184307 expression
no PRJNA562324 expression
no GSE172372 expression
no GSE215144 expression
no GSE217012 expressio

(20062, 23440)

In [ ]:
map_raw = {'Main4_Th1_signature':['Th1_cells'],
'Main4_CD8_T_cells':['CD8_T_cells'],
'Main4_Treg':['Tregs'],
'Main4_Neutrophil_signature':['Neutrophils'],
'Main4_Mast_cell_signature':['Mast_cells'],
'Main4_Effector_cells':['CD8_T_cells', 'NK_cells'],
'Main4_Eosinophil_signature':['Eosinophils'],
'Main4_Follicular_helper_T_cells':['Follicular_T_helper_tonsil'],
'Main4_B_cells': ['B_cells'],
'Main4_Endothelium':['Endothelium'],
'Main4_Pan_macrophage_signature':['Macrophages'],
'Main4_NK_cells':['NK_cells'],
'Main4_M1_signatures':['Macrophages_M1'],
'Main4_M2_signature':['Macrophages_M2'],
'Main4_T_cells':['T_cells'],
'Main4_CD4_T_cells':['CD4_T_cells','CD4_T_helpers'],
'Main4_Lymphatic_endothelium':['Endothelium_lymph'],
'Main4_Th17_signature':['Th17_cells'],
'Main4_Plasma_cells':['Plasma_B_cells', 'Plasmablasts'],
'Main4_Monocyte':['Monocytes'],
}

controls = list(cells_clean['Cell_type'].value_counts()[cells_clean['Cell_type'].value_counts()>50].index)
controls = [i for i in controls if i !='Undefined']
gois = [map_raw[sign][0] for sign in  map_raw.keys()]
controls_order = [
 'T_cells',
 'CD4_T_helpers',
 'CD4_T_cells',
     'PD1_CD4_T_cells',
 'Memory_CD4_T_cells',
     'Th1_cells',
   'Th2_cells',
    'Th2',
    'Th17_cells',
    'Follicular_T_helper_tonsil',
 'Tregs',
 'CD8_T_cells',
    'Memory_CD8_T_cells',
     'CD8_T_cells_PD1_high',
 'NK_cells',
 'B_cells',
'Plasma_B_cells',
'Plasmablasts',
 'Non_plasma_B_cells',
 'Myeloid_cells',
 'Neutrophils',
 'Eosinophils',
    'Mast_cells',
 'Monocytes',
 'Macrophages',
     'Macrophages_M1',
 'Macrophages_M2',
 'Monocytic_DC',
    'Dendritic_cells',
   'Fibroblasts',
 'Cardiac_myofibroblasts',
 'Endothelium',
    'Endothelium_lymph',
 'Hepatocytes',-0
'Astrocytes',
 'Bronchial_cells',
 'Epithelium',
 'Fibroblast_line',
 'Follicular_T_helper',
 'Keratinocytes',
 'MAIT_cells',
 'MSC',
 'Neurons',
 'Pancreatic_cells',
 'iPSC'
]
controls = list(set(controls_order+gois))

mapping = {sign:{'Goi':map_raw[sign],'Control':[i for i in controls],'Deleted_controls':[]} for sign in map_raw.keys()}

controls_to_delete = {'Main4_Th1_signature':['T_cells', 'CD4_T_cells', 'CD4_T_helpers', 'Memory_CD4_T_cells'],
'Main4_CD8_T_cells':['T_cells','Memory_CD8_T_cells',     'CD8_T_cells_PD1_high', 'MAIT_cells'],
'Main4_Treg':['T_cells', 'CD4_T_cells','Memory_CD4_T_cells'],
'Main4_Neutrophil_signature': ['Myeloid_cells'],
'Main4_Mast_cell_signature': ['Myeloid_cells'],
'Main4_Effector_cells':['T_cells'],
'Main4_Eosinophil_signature': ['Myeloid_cells'],
'Main4_Follicular_helper_T_cells':['T_cells','CD4_T_cells','T_cells','CD4_T_helpers','Memory_CD4_T_cells',],
'Main4_B_cells': ['Plasma_B_cells', 'Non_plasma_B_cells','Plasmablasts'],
'Main4_Endothelium':['Endothelium_lymph'],
'Main4_Pan_macrophage_signature':['Macrophages_M1', 'Macrophages_M2', 'Myeloid_cells','Monocytes'],
'Main4_NK_cells':[],
'Main4_M1_signatures':['Macrophages', 'Myeloid_cells', 'Monocytes'],
'Main4_M2_signature':['Macrophages', 'Myeloid_cells', 'Monocytes'],
'Main4_T_cells': [ 'CD4_T_helpers', 'CD4_T_cells',     'PD1_CD4_T_cells',
 'Memory_CD4_T_cells',     'Th1_cells',    'Th17_cells','Th2_cells',
    'Th2',
    'Follicular_T_helper_tonsil', 'Tregs', 'CD8_T_cells',
    'Memory_CD8_T_cells',     'CD8_T_cells_PD1_high',],
'Main4_CD4_T_cells':['PD1_CD4_T_cells','T_cells','Th1_cells',    'Th17_cells','Follicular_T_helper_tonsil', 'Tregs'],
'Main4_Lymphatic_endothelium':['Endothelium'],
'Main4_Th17_signature':['T_cells', 'CD4_T_cells', 'CD4_T_helpers', 'Memory_CD4_T_cells'],
'Main4_Plasma_cells':['B_cells'],
'Main4_Monocyte':['Macrophages', 'Myeloid_cells', 'Macrophages_M1', 'Macrophages_M2','Monocytic_DC'],
}

for sign in mapping.keys():
    to_delete = list(set(controls_to_delete[sign]+map_raw[sign]))
    x = len(mapping[sign]['Control'])
    for element_to_remove in to_delete:
        mapping[sign]['Control'].remove(element_to_remove)
        mapping[sign]['Deleted_controls'].append(element_to_remove)
    print(sign,x,len(mapping[sign]['Control']))

mapping_ssgseas = {sign: {'Goi': {ct:[] for ct in mapping[sign]['Goi']},  'Control': {ct:[] for ct in mapping[sign]['Control']},
                         'Deleted_controls': {ct:[] for ct in mapping[sign]['Deleted_controls']}} for sign in mapping.keys()}

for sign in mapping.keys():
        for group in ['Goi','Control','Deleted_controls']:
            for ct in mapping[sign][group]:
                part = public_cells_expr[public_cells_annot[public_cells_annot.Cell_type.isin([ct])].index]
                x = ssgsea_formula(part, gmt_genes_alt_names(msigdb_gmt[sign],public_cells_expr.index)).T
                mapping_ssgseas[sign][group][ct] = x
                print(sign, ct, x.shape)              

import pickle
with open('/home/jovyan/SignValArticle/mapping_ssgseas.pkl','wb') as handle:
    pickle.dump(mapping_ssgseas,handle,pickle.HIGHEST_PROTOCOL)
import pickle
with open('/home/jovyan/SignValArticle/mapping_ssgseas.pkl','rb') as handle:
    mapping_ssgseas = pickle.load(handle)

for key in mapping_ssgseas.keys():
    for i in mapping_ssgseas[key]['Goi'].keys():
        print(key,'\t',i,'\t', mapping_ssgseas[key]['Goi'][i].shape)

NameError: name 'cells_clean' is not defined

In [ ]:
V1_GMT_PICKLE = "/home/jovyan/projects/SignVal/Signature_validation/article/msigdb_gmt.pkl"

v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
in_scope_fges = [k for k in MAP_RAW if k not in EXCLUDED_FGES_RARE]
v1_gmt = select_msigdb_gmt_subset(v1_gmt_full, in_scope_fges)

for sign in in_scope_fges:
    assert sign in v1_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    n_random = count_random_fges(v1_gmt[sign])
    assert n_random == 10, f"{sign}: expected 10 RANDOM_FGES, got {n_random}"

msigdb_gmt = harmonize_gmt_to_index(v1_gmt, public_cells_expr.index)
logger.info(
    "msigdb_gmt: {} FGES, {} signatures total",
    len(msigdb_gmt),
    sum(len(v) for v in msigdb_gmt.values()),
)

ModuleNotFoundError: No module named 'bioreactor'

In [ ]:
mapping = build_mapping(annotation=public_cells_annot)
controls_present = intersect_controls_with_cohort(CONTROLS_ORDER, public_cells_annot)
logger.info(
    "in-scope: {} FGES; controls present in new cohort: {}",
    len(mapping),
    len(controls_present),
)
for sign, bucket in mapping.items():
    logger.info(
        "{}: GOI={}, Control={}, Deleted={}",
        sign,
        bucket["Goi"],
        len(bucket["Control"]),
        len(bucket["Deleted_controls"]),
    )

In [ ]:
mapping_ssgseas = compute_mapping_ssgseas(
    public_cells_expr=public_cells_expr,
    public_cells_annot=public_cells_annot,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
)

for sign in mapping_ssgseas:
    if sign in EXCLUDED_FGES_RARE:
        continue
    assert mapping_ssgseas[sign]["Goi"], f"{sign}: GOI cohort is empty"
    for ct, frame in mapping_ssgseas[sign]["Goi"].items():
        logger.info("{} GOI {}: {} samples", sign, ct, frame.shape[0])

with open(MAPPING_SSGSEAS_PATH, "wb") as fh:
    pickle.dump(mapping_ssgseas, fh, pickle.HIGHEST_PROTOCOL)
logger.info("wrote {}", MAPPING_SSGSEAS_PATH)

In [ ]:
out = compute_out_table(mapping_ssgseas, mapping, msigdb_gmt, controls_present)
out = fdr_correct_out(out, controls_present)
out.to_csv(OUT_TSV_PATH, sep="\t")
logger.info("wrote {} ({} rows × {} cols)", OUT_TSV_PATH, *out.shape)
out.head()

In [ ]:
plot_violin_per_source(mapping_ssgseas, save_dir=OUTPUT_DIR)
plot_signature_heatmap(
    mapping_ssgseas=mapping_ssgseas,
    out_df=out,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
    annotation=public_cells_annot,
    controls_order=controls_present,
    palette={ct: cells_p[ct] for ct in controls_present if ct in cells_p},
    save_path=HEATMAP_PATH,
    short=True,
)
averaged = plot_sens_spec_scatter(
    mapping_ssgseas=mapping_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
)
logger.info("plots saved under {}", OUTPUT_DIR)

## Rare cell types (out of scope here)

FGES whose GOI is rare in the new cohort — `Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` — are deferred to a separate notebook authored by Nadezhda. That notebook reuses the original cohort (`/uftp2/.../cells_all_annotation.tsv` plus the Tonsillar Tfh / Mast / Endothelium_lymph patches), generates 10 stratified 75/25 holdouts via `signature_validation.benchmark.splits.stratified_holdout_indices` (stratified by BG-FGES score median × GOI/Control), scores ssGSEA on each test fold and aggregates with `aggregate_score_over_splits`. Rare cell types are starred on the resulting figures.